## Exploratory Data Analysis & Feature Engineering

Required imports for sending API calls and parsing the data

In [3]:
import re
import time
import random
import pandas as pd
import urllib.request as libreq
from pypdf import PdfReader
from urllib.parse import urlencode
from pathlib import Path
from datetime import datetime, timezone
from urllib.parse import urlsplit
from urllib.error import HTTPError



In [4]:
feed_url = "https://rss.arxiv.org/atom/cs.AI+cs.LG"

with libreq.urlopen(feed_url, timeout=60) as response:
    feed_data = response.read()

dataset_dir = Path("arxiv_data")
dataset_dir.mkdir(exist_ok=True)

timestamp = datetime.now(timezone.utc).strftime("%Y%m%dT%H%M%SZ")
feed_path = dataset_dir / f"cs_ai_feed_{timestamp}.xml"
feed_path.write_bytes(feed_data)

print(f"Saved feed to: {feed_path}")

Saved feed to: arxiv_data/cs_ai_feed_20260916T024206Z.xml


This cell saves the data from the XML as a JSON

In [5]:
import json
import xml.etree.ElementTree as ET

ns = {"atom": "http://www.w3.org/2005/Atom"}
root = ET.fromstring(feed_data)

papers = []

for entry in root.findall("atom:entry", ns):
    paper = {
        "id": entry.findtext("atom:id", "", ns),
        "title": " ".join(
            entry.findtext("atom:title", "", ns).split()
        ),
        "abstract": entry.findtext("atom:summary", "", ns).strip(),
        "authors": [
            author.findtext("atom:name", "", ns)
            for author in entry.findall("atom:author", ns)
        ],
        "published": entry.findtext("atom:published", "", ns),
        "updated": entry.findtext("atom:updated", "", ns),
        "source_url": next(
            (
                link.attrib.get("href", "")
                for link in entry.findall("atom:link", ns)
                if link.attrib.get("rel", "alternate") == "alternate"
            ),
            "",
        ),
    }
    papers.append(paper)

metadata_path = feed_path.with_suffix(".json")
metadata_path.write_text(
    json.dumps(papers, indent=2, ensure_ascii=False),
    encoding="utf-8",
)

print(f"Saved {len(papers)} announcements to: {metadata_path}\n")

for paper in papers[:10]:
    print(paper["title"])
    print(paper["source_url"])
    print()

Saved 989 announcements to: arxiv_data/cs_ai_feed_20260916T024206Z.json

BudgetBench: A Budget-Tiered Protocol and Pilot Harness for Memory Strategy Evaluation in Local Large Language Model Agents
https://arxiv.org/abs/2609.13149

A derivative-fidelity failure mode in physics-informed neural networks: strengthened benchmark evidence from function-value training
https://arxiv.org/abs/2609.13171

Land Art as a Big-Data Climate Sensor
https://arxiv.org/abs/2609.13182

LLMs or Naive Bayes? Old Gems or New Ways
https://arxiv.org/abs/2609.13185

Early Prediction of Satellite Collision Probability Using a Hybrid TCN-Transformer Model for a CDM-Based Conjunction Analysis Framework
https://arxiv.org/abs/2609.13191

Evaluating LLM-Generated Rules for Heart Disease Prediction
https://arxiv.org/abs/2609.13192

Diagnosing Faults in Reinforcement Learning Simulators and World Models with Canonical Polynomial Invariants
https://arxiv.org/abs/2609.13194

Machine Unlearning for Speech Question Answerin

In [6]:
dataset_dir = Path("arxiv_data")
pdf_dir = dataset_dir / "pdfs"
pdf_dir.mkdir(parents=True, exist_ok=True)

# get a random 100 papers from the list of papers
selected_papers = random.sample(papers, min(100, len(papers)))
storage_limit = 700_000_000    # 700 MB, leaving room below 1 GB

def folder_size():
    return sum(
        file.stat().st_size
        for file in dataset_dir.rglob("*")
        if file.is_file()
    )

for paper in selected_papers:
    # Extract the paper ID from its arXiv abstract-page link.
    path = urlsplit(paper["source_url"]).path
    match = re.fullmatch(r"/abs/(\d{4}\.\d{4,5}(?:v\d+)?)", path)

    if not match:
        print("Skipping unrecognized link:", paper["source_url"])
        continue

    paper_id = match.group(1)
    pdf_url = f"https://export.arxiv.org/pdf/{paper_id}"
    pdf_path = pdf_dir / f"{paper_id}.pdf"
    temp_path = pdf_path.with_suffix(".part")

    if pdf_path.exists():
        print("Already downloaded:", pdf_path.name)
        continue

    temp_path.unlink(missing_ok=True)
    available_bytes = storage_limit - folder_size()

    if available_bytes <= 0:
        print("Storage limit reached. Stopping.")
        break

    print("Downloading:", paper["title"], flush=True)
    time.sleep(3.1)
    downloaded_bytes = 0

    try:
        with libreq.urlopen(pdf_url, timeout=60) as response:
            with temp_path.open("wb") as file:
                while True:
                    chunk = response.read(64 * 1024)
                    if not chunk:
                        break

                    if downloaded_bytes == 0 and not chunk.startswith(b"%PDF-"):
                        raise ValueError("Server response was not a PDF.")

                    if downloaded_bytes + len(chunk) > available_bytes:
                        raise ValueError("Download would exceed 700 MB.")

                    file.write(chunk)
                    downloaded_bytes += len(chunk)

        if downloaded_bytes == 0:
            raise ValueError("Server returned an empty file.")

        temp_path.replace(pdf_path)
        print(f"Saved {pdf_path.name}: {downloaded_bytes / 1_000_000:.2f} MB")

    except HTTPError as error:
        print(f"HTTP {error.code}: stopping downloads.")
        print(error.read().decode("utf-8", errors="replace")[:300])
        error.close()
        break

    except (OSError, ValueError) as error:
        print("Stopped:", error)
        break

    finally:
        temp_path.unlink(missing_ok=True)

print(f"\nDataset folder size: {folder_size() / 1_000_000:.2f} MB")
print("PDF location:", pdf_dir.resolve())

Downloading: Context-Dependent Affordance Reports in Vision-Language Models
Saved 2603.04419.pdf: 0.25 MB
Downloading: A Language-Guided Multimodal Foundation Model for Zero-Shot and Multi-Task Brain Signal Analysis
Saved 2609.15740.pdf: 9.47 MB
Downloading: Internalize the Temperature: On-Policy Self-Distillation as Policy Reheater for Reinforcement Learning
Saved 2606.00755.pdf: 0.99 MB
Downloading: MMLA: Memory-Mediated Learning Architecture for Predictive Dual-State Adaptation
Saved 2606.28876.pdf: 2.50 MB
Downloading: MAST: Label-Efficient, Robust, and Generalizable Sound Detection for Biodiversity Monitoring via Masked Audio Pretraining and Self-Training
Saved 2609.15221.pdf: 2.40 MB
Downloading: The Normalization of Deviance in AI Development
Saved 2609.05749.pdf: 0.14 MB
Downloading: FaithfulBench: Does AI Counsel Uphold or Undermine the User's Professed Faith?
Saved 2609.13634.pdf: 1.34 MB
Downloading: Not All Prompts Are Equal: Exploration-Guided Prompt Scaffolding for Multim

In [9]:
from pathlib import Path

pdf_dir = Path("arxiv_data/pdfs")
pdfs = sorted(pdf_dir.glob("*.pdf"))

total_bytes = sum(pdf.stat().st_size for pdf in pdfs)

for pdf in pdfs:
    print(f"{pdf.name}: {pdf.stat().st_size / 1_000_000:.2f} MB")

print(f"\nPDF count: {len(pdfs)}")
print(f"Total PDF size: {total_bytes / 1_000_000:.2f} MB")
print(f"Total PDF size: {total_bytes / 1_000_000_000:.4f} GB")

if pdfs:
    average_mb = total_bytes / len(pdfs) / 1_000_000
    print(f"Average per PDF: {average_mb:.2f} MB")

2404.07729.pdf: 2.10 MB
2410.24205.pdf: 3.89 MB
2501.03008.pdf: 0.70 MB
2503.00389.pdf: 1.05 MB
2504.01157.pdf: 0.77 MB
2506.04166.pdf: 0.91 MB
2509.00303.pdf: 2.41 MB
2511.09665.pdf: 0.42 MB
2512.08463.pdf: 19.97 MB
2512.24999.pdf: 1.05 MB
2601.00900.pdf: 3.56 MB
2602.13940.pdf: 8.39 MB
2602.21061.pdf: 0.76 MB
2602.24289.pdf: 5.79 MB
2603.04419.pdf: 0.25 MB
2603.18136.pdf: 1.46 MB
2603.23433.pdf: 1.13 MB
2605.01777.pdf: 2.01 MB
2605.27840.pdf: 0.61 MB
2605.28916.pdf: 1.05 MB
2605.31291.pdf: 1.98 MB
2606.00755.pdf: 0.99 MB
2606.01008.pdf: 0.66 MB
2606.06379.pdf: 3.52 MB
2606.07908.pdf: 0.48 MB
2606.10829.pdf: 0.42 MB
2606.14397.pdf: 11.53 MB
2606.17995.pdf: 3.11 MB
2606.28876.pdf: 2.50 MB
2606.31285.pdf: 0.85 MB
2607.01043.pdf: 1.05 MB
2607.14193.pdf: 2.08 MB
2607.17425.pdf: 0.59 MB
2609.03846.pdf: 1.05 MB
2609.05749.pdf: 0.14 MB
2609.12018.pdf: 4.34 MB
2609.13206.pdf: 20.81 MB
2609.13225.pdf: 0.13 MB
2609.13242.pdf: 1.29 MB
2609.13243.pdf: 8.37 MB
2609.13247.pdf: 2.55 MB
2609.13272.pd

This cell checks for any files that might have failed to download properly

In [14]:

checks = []

for pdf in sorted(Path("arxiv_data/pdfs").glob("*.pdf")):
    size = pdf.stat().st_size

    with pdf.open("rb") as file:
        header = file.read(5)
        file.seek(max(0, size - 4096))
        ending = file.read()

    checks.append({
        "filename": pdf.name,
        "size_mb": size / 1_000_000,
        "pdf_header": header == b"%PDF-",
        "eof_near_end": b"%%EOF" in ending,
    })

pdf_checks = pd.DataFrame(
    checks,
    columns=["filename", "size_mb", "pdf_header", "eof_near_end"],
)

suspect_files = pdf_checks[
    (pdf_checks["pdf_header"] == False)
    | (pdf_checks["eof_near_end"] == False)
]

display(suspect_files)

,filename,size_mb,pdf_header,eof_near_end


The cell below removes any suspicious files. Only run if the above cell found any mismatches

In [ ]:
pdf_dir = dataset_dir / "pdfs"

# Record what was excluded and why.
suspect_files.assign(
    exclusion_reason="Missing PDF header or ending marker"
).to_csv(dataset_dir / "excluded_pdfs.csv", index=False)

removed = 0

for filename in suspect_files["filename"]:
    pdf_path = pdf_dir / filename

    if pdf_path.is_file():
        pdf_path.unlink()
        removed += 1
        print("Deleted:", filename)

print(f"\nRemoved {removed} PDFs.")

This cell gives some basic information about the PDFs as well as data about the collection of papers

In [15]:
pdf_dir = Path("arxiv_data/pdfs")

rows = []
pages_by_paper = {}

for pdf in sorted(pdf_dir.glob("*.pdf")):
    row = {
        "filename": pdf.name,
        "size_mb": pdf.stat().st_size / 1_000_000,
        "pages": None,
        "approx_words": None,
        "low_text_pages": None,
        "extraction_error": "",
    }

    try:
        reader = PdfReader(pdf)
        page_texts = [page.extract_text() or "" for page in reader.pages]

        # Keep page boundaries for inspection and future citations.
        pages_by_paper[pdf.name] = page_texts

        row.update({
            "pages": len(page_texts),
            "approx_words": sum(len(text.split()) for text in page_texts),
            "low_text_pages": sum(
                len(text.strip()) < 100 for text in page_texts
            ),
        })

    except Exception as error:
        row["extraction_error"] = str(error)

    rows.append(row)

eda = pd.DataFrame(rows)

display(eda)
display(eda[["size_mb", "pages", "approx_words"]].describe())

eda.to_csv("arxiv_data/paper_eda.csv", index=False)

Exceeded 5000 form XObject invocations while extracting text; further form content is skipped.
Exceeded 5000 form XObject invocations while extracting text; further form content is skipped.


,filename,size_mb,pages,approx_words,low_text_pages,extraction_error
0,2410.24205.pdf,3.893414,8,6813,0,
1,2501.03008.pdf,0.704458,9,4819,0,
2,2504.01157.pdf,0.771202,4,3104,0,
3,2506.04166.pdf,0.907062,21,9673,0,
4,2509.00303.pdf,2.411425,15,12909,0,
...,...,...,...,...,...,...
96,2609.15723.pdf,0.894468,57,20127,0,
97,2609.15740.pdf,9.469725,38,15501,0,
98,2609.15773.pdf,2.940938,13,8887,0,
99,2609.15871.pdf,1.381421,6,5064,0,


,size_mb,pages,approx_words
count,101.000000,101.000000,101.000000
mean,3.465583,22.683168,10814.594059
std,6.298683,24.221449,11578.858332
min,0.126627,4.000000,2676.000000
25%,0.596746,10.000000,5092.000000
50%,1.129014,16.000000,9149.000000
75%,2.953236,30.000000,13851.000000
max,39.539815,222.000000,112203.000000
